# Autoencoder

## 0. Imports

In [12]:
import os
import sys

# Get the absolute path of the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add the parent directory to sys.path if it's not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [ ]:
import itertools
from dataclasses import dataclass
from datetime import datetime

import pandas as pd
from pathlib import Path
from tensorflow.keras.models import Model  # type: ignore
from tensorflow.keras.layers import (  # type: ignore
    Input,
    Dense,
    Dropout,
    BatchNormalization,
    Activation,
)
from tensorflow.keras.optimizers import Adam  # type: ignore
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau  # type: ignore
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import evaluation as evals
import plotter as plotter

## 1. Load Data

In [3]:
data_dir = Path("../../data/")

# read features
features_path = data_dir / "features_df.csv"
features_df = pd.read_csv(f"{features_path}")
print(f"Features Shape: {features_df.shape}")

Features Shape: (1932, 110)


In [4]:
# optionally read the edgelist in for now
edgelist_path = data_dir / "edgelists" / "edgelist_full.csv"
edgelist = pd.read_csv(f"{edgelist_path}")
print(f"Edgelist Shape: {edgelist.shape}")

Edgelist Shape: (3730692, 3)


Some quick validations:

- Number of users in each?

In [5]:
print(
    f"Unique user count in features_df vs. edgelist: \
    {features_df.user_id.nunique()} \
    {edgelist.user_anchor.nunique()}"
)
feature_df_users = set(features_df.user_id.tolist())
edgelist_users = set(edgelist.user_anchor.tolist())
print(f"Intersection length = {len(feature_df_users.intersection(edgelist_users))}")

Unique user count in features_df vs. edgelist:     1932     1932
Intersection length = 1932


## 2. Pre-processing

In [6]:
# save user ids for mapping
user_ids = features_df["user_id"].values

# one-hot encode all categorical
cat_cols = features_df.select_dtypes(include=["object", "bool"]).columns.tolist()
if "user_id" in cat_cols:
    cat_cols.remove("user_id")
df_numeric = features_df.drop(columns=["user_id"])
df_numeric = pd.get_dummies(df_numeric, columns=cat_cols)
print("One-hot encoding complete")

# fillna with median
df_numeric = df_numeric.fillna(df_numeric.median())
print("Filled missing with median")

# standardize and construct the X matrix
scaler = StandardScaler()
X = scaler.fit_transform(df_numeric)
print(X.shape)

X_train, X_test, ids_train, ids_test = train_test_split(
    X, user_ids, test_size=0.2, random_state=42
)

print(f"Train Shape: {X_train.shape}, Test Shape: {X_test.shape}")

One-hot encoding complete
Filled missing with median
(1932, 116)
Train Shape: (1545, 116), Test Shape: (387, 116)


/var/folders/jh/9_qy7nd96v9_q0_x1sffyq9c0000gn/T/ipykernel_45811/2416148417.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = features_df.select_dtypes(include=["object", "bool"]).columns.tolist()


## 3. Training Autoencoder

### Autoencoder Factory

Here's a general class implementation for the autoencoder that we can reuse

In [7]:
@dataclass
class Config:
    encoding_dim: int
    layers: list[int]
    learning_rate: float
    dropout_rate: float
    patience: int
    bottleneck_activation: str
    batch_size: int

    def __str__(self) -> str:
        parts = [
            f"model_enc{self.encoding_dim}",
            f"dep{len(self.layers)}",
            f"lr{self.learning_rate}",
            f"pat{self.patience}",
            f"drop{self.dropout_rate}",
            f"batch{self.batch_size}",
            f"bn_act-{self.bottleneck_activation}",
        ]
        return "_".join(parts)


class AutoencoderBuilder:
    """Factory for creating autoencoders"""

    def __init__(self, input_dim, config: list[Config]):
        self.input_dim = input_dim
        self.config = config
        self.model = self._build_model()

    def _build_model(self):
        inputs = Input(shape=(self.input_dim,))

        # ENCODER
        x = inputs
        for units in self.config.layers:
            x = Dense(units)(x)
            x = BatchNormalization()(x)
            x = Activation("relu")(x)
            x = Dropout(self.config.dropout_rate)(x)

        # Bottleneck (latent space)
        x = Dense(self.config.encoding_dim)(x)
        x = BatchNormalization()(x)
        encoded = Activation(
            self.config.bottleneck_activation, name="bottleneck_output"
        )(x)

        # DECODER
        x = encoded
        for units in reversed(self.config.layers):
            x = Dense(units)(x)
            x = BatchNormalization()(x)
            x = Activation("relu")(x)
            x = Dropout(self.config.dropout_rate)(x)

        # Final reconstruction – no bottleneck or anything
        decoded = Dense(self.input_dim, activation="linear")(x)

        # Assemble
        autoencoder = Model(inputs, decoded)
        optimizer = Adam(learning_rate=self.config.learning_rate)
        autoencoder.compile(optimizer=optimizer, loss="mse")

        return autoencoder

    def get_encoder(self):
        """This creates a new model that stops at the 'encoded' layer
        We grab the layer by name or by index (0 is input, then dense, bn, act...)
        Finding by name is safest"""
        encoder_output = self.model.get_layer("bottleneck_output").output
        return Model(inputs=self.model.input, outputs=encoder_output)

### Hyperparameters

Below parameters don't need to be tuned

In [8]:
loss = "mse"
epochs = 100
batch_size = 32
validation_split = 0.2

These parameters can be tuned. I put them inside a `Config` class:

```python
@dataclass
class Config:
    encoding_dim: int
    layers: list[int]
    learning_rate: float
    dropout_rate: float
    patience: int
```

Learning Rate, Patience, and Dropout are regularizers, so more important if we find
a good model that isn't generalizing. The other two – encoding dimension and layers,
are critical to find a model structure that works

Here's the things we can basically change:

| Hyperparameter | What it does | Suggested Experiment | 
| --- | --- | ---
| Encoding Dim | The size of the "bottleneck." | If reconstruction loss is high, the bottleneck might be too tight. Try bumping 32 up to 48 or 64. |
| Depth | Adding more layers. | Try adding another layer (e.g., 128→64→32) to help the model learn more complex hierarchical features. | 

### Training Loop

Create the hyperparameter grid across wwhich we will search 

In [ ]:
# The configuration grid
grid = {
    "encoding_dim": [32, 48, 64, 96],
    "layers": [[64], [256, 128], [512, 256, 128]],
    "learning_rate": [0.001, 0.0025, 0.005],
    "dropout_rate": [0.1, 0.2],
    "patience": [10, 20],
    "batch_size": [32, 256],
    "bottleneck_activation": ["relu", "linear"],
}

# generates all combinations
keys = grid.keys()
values = grid.values()
all_configs = [Config(**dict(zip(keys, v))) for v in itertools.product(*values)]

# NOTE: important bit that filters out invalid configurations
# suppose encoding_dim = 96, but I had only one layer with 64 neurons
# the problem is my encoder reduces input_dim -> 64 -> 96...that second transformation
# is redundant
valid_configs = []
for config in all_configs:
    # The last layer in the encoder sequence
    last_hidden_layer_size = config.layers[-1]

    # Only keep it if it shrinks into the bottleneck
    if config.encoding_dim < last_hidden_layer_size:
        valid_configs.append(config)

print(f"Generated {len(all_configs)} total combinations.")
print(f"Filtered down to {len(valid_configs)} valid funnel architectures.")

Run the training loop

In [ ]:
# for saving outputs
run_id = datetime.now().strftime("%Y%m%d_%H%M")
os.makedirs(f"experiments/{run_id}", exist_ok=True)

results = []

for i, config in enumerate(valid_configs):
    print(f"Running experiment {i + 1}/{len(valid_configs)}")

    # SETUP
    builder = AutoencoderBuilder(input_dim=X.shape[1], config=config)

    # Early Stopping
    early_stop = EarlyStopping(
        monitor="val_loss", patience=config.patience, restore_best_weights=True
    )

    # LR Scheduler
    # If loss plateaus for 3 epochs, multiply LR by 0.5
    lr_schedule = ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6
    )

    # TRAIN
    history = builder.model.fit(
        X_train,
        X_train,
        epochs=30,
        batch_size=config.batch_size,
        validation_split=0.2,
        verbose=0,
    )

    # SAVE
    model_name = str(config)
    model_path = f"experiments/{run_id}/{model_name}.keras"
    builder.model.save(model_path)

    # EVALUATION

    # get validation MSE
    val_mse = history.history["val_loss"][-1]

    # get test set MSE
    test_mse = builder.model.evaluate(X_test, X_test, verbose=0)

    # get the encoder, and run our metrics
    encoder = builder.get_encoder()
    embeddings = encoder.predict(X_test, verbose=0)

    # evaluator
    evaluator = evals.RecommenderEvaluator(embeddings, ids_test, edgelist)
    metrics = evaluator.get_all_metrics()

    # Flatten config into a dict for the DataFrame
    res_dict = {
        "name": model_name,
        "encoding_dim": config.encoding_dim,
        "depth": len(config.layers),
        "lr": config.learning_rate,
        "patience": config.patience,
        "dropout_rate": config.dropout_rate,
        "val_mse": val_mse,
        "test_mse": test_mse,
        **metrics,
    }
    results.append(res_dict)

# Analyze
results_df = pd.DataFrame(results).sort_values(by="val_mse")
results_df.to_csv(f"experiments/{run_id}/manifest.csv", index=False)